## 20. Claude Code 文件系统特性：setting_sources 是总开关

> 来源：[Use Claude Code features in the SDK](https://code.claude.com/docs/en/agent-sdk/claude-code-features)、[Agent Skills in the SDK](https://code.claude.com/docs/en/agent-sdk/skills)、[Slash Commands in the SDK](https://code.claude.com/docs/en/agent-sdk/slash-commands)、[Plugins in the SDK](https://code.claude.com/docs/en/agent-sdk/plugins)

SDK 与 Claude Code 同底座，CLI 的文件系统配置（CLAUDE.md、rules、skills、hooks、settings、slash commands、output styles）SDK 都能加载，总开关是 `setting_sources`。


### 20.1 三个 source 的语义

**省略 `setting_sources` = 全加载（`["user", "project", "local"]`），与 CLI 行为一致；传 `[]` = 只认代码配置。**

| Source | 加载什么 | 位置要点 |
|---|---|---|
| `"project"` | 项目 CLAUDE.md、`.claude/rules/*.md`、项目 skills、项目 hooks、`.claude/settings.json`、`.mcp.json` | CLAUDE.md/rules 从 `<cwd>` 回溯**所有**父目录；skills 回溯到 repo root 为止；settings.json/hooks **只看** `<cwd>/.claude/` |
| `"user"` | `~/.claude/CLAUDE.md`、用户 rules/skills/settings | `~/.claude/` |
| `"local"` | `CLAUDE.local.md`、`.claude/settings.local.json` | 本机私有配置 |

拿一个具体目录布局套一遍语义（`cwd = ~/repo/app`）：

```text
~/.claude/CLAUDE.md               ← "user"
~/repo/CLAUDE.md                  ← "project"（父目录也回溯）
~/repo/app/CLAUDE.md              ← "project"
~/repo/app/sub/CLAUDE.md          ← "project"（子目录按需：agent 读到该子树的文件时才加载）
~/repo/app/.claude/settings.json  ← "project"（settings 只看 cwd 下的 .claude/，~/repo/.claude/ 里的不算）
~/repo/app/CLAUDE.local.md        ← "local"

# setting_sources 省略        → 全部加载
# setting_sources=["project"] → 只加载中间四个
# setting_sources=[]          → 全不读，agent 只认代码里传的配置
```

各级 CLAUDE.md **叠加生效**，没有硬性优先级；指令冲突时结果取决于 Claude 的解读，官方建议写不冲突的规则。

> [!warning] setting_sources 管不住的四个输入（多租户必读)
> ① Managed policy settings（MDM/注册表/server-managed）始终加载；② `~/.claude.json` 全局配置始终读取（用 `env={"CLAUDE_CONFIG_DIR": ...}` 重定位）；③ Auto memory 会载入 system prompt（`CLAUDE_CODE_DISABLE_AUTO_MEMORY=1` 关闭）；④ claude.ai MCP connectors（订阅认证时）`mcp_servers={}` 压不掉。**多租户部署 = 每租户独立文件系统 + `setting_sources=[]` + 禁 auto memory**，不能只靠默认选项隔离。

`"project"` 加载的 hooks 是 `settings.json` 里的**文件式 hooks**（shell 命令），它们与 `query()` 传入的**编程式 hooks** 并行运行、同属一个生命周期——CLI 交互会话配好的钩子，SDK 里零配置直接生效。下方 cell 是官方示例，两者同时上：`setting_sources` 加载文件系统配置，编程式 `PreToolUse` hook 拦截危险命令：

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, HookMatcher, ResultMessage


# 编程式 hook：拦下危险命令。返回 {} 表示放行；
# deny 走 hookSpecificOutput（顶层 decision/reason 写法已废弃）
async def audit_bash(input_data, tool_use_id, context):
    command = input_data.get("tool_input", {}).get("command", "")
    if "rm -rf" in command:
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": "Destructive command blocked",
            }
        }
    return {}


async def demo_setting_sources():
    async for message in query(
        prompt="Help me refactor the auth module",
        options=ClaudeAgentOptions(
            # "user" 读 ~/.claude/，"project" 读 cwd 的 .claude/——
            # CLAUDE.md、skills、文件式 hooks、permissions 一并加载
            setting_sources=["user", "project"],
            allowed_tools=["Read", "Edit", "Bash"],
            # 编程式 hooks 与 settings.json 里的文件式 hooks 并行运行
            hooks={"PreToolUse": [HookMatcher(matcher="Bash", hooks=[audit_bash])]},
        ),
    ):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_setting_sources()

### 20.2 Skills

Skill = `.claude/skills/<name>/SKILL.md`（YAML frontmatter + Markdown 正文），Claude 依据 `description` **自主调用**；SDK 没有编程注册 Skills 的 API（与子 agent 相反），只能走文件系统。一个最小 skill 文件：

```markdown
# .claude/skills/changelog/SKILL.md
---
name: changelog
description: 生成版本变更日志。用户要求整理 changelog 或 release notes 时使用。
---

读取 git log 与现有 CHANGELOG.md，按 Keep a Changelog 格式输出本次变更……
```

启动时只发现 metadata（上面的 frontmatter），触发时才加载正文全文，所以对 context 很省。加载验证：init 消息的 `data["skills"]` 是本 session 可用的 skill 名清单——真实 Claude Code 会话的 init 消息里它长这样：`'skills': ['ai-learning', 'algorithm-playbook', 'archimate', ...]`。

`skills` 选项做会话内过滤：省略/`"all"` = 全部启用；`["pdf", "docx"]` = 只启用列出的；`[]` = 全禁。设了 `skills` 时 SDK 自动把 `Skill` tool 加进 allowed 名单，但若同时传了 `tools` 收窄名单，要自己把 `"Skill"` 加回去。

两个坑：`skills` 是 **context 过滤器不是沙箱**——没启用的 skill 文件还在磁盘上，Read/Bash 照样碰得到；SKILL.md 的 `allowed-tools` frontmatter **只在 CLI 生效**，SDK 下工具权限一律由主 `allowed_tools` 控制。

官方示例见下方 cell：发现并全量启用项目 skills，让 Claude 按项目里的 code review checklist 审 PR：

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage


async def demo_skills():
    # setting_sources 含 "project" 时，.claude/skills/ 下的 skill 自动被发现
    async for message in query(
        prompt="Review this PR using our code review checklist",
        options=ClaudeAgentOptions(
            setting_sources=["user", "project"],
            skills="all",                           # 全量启用；也可传名字列表，[] 为全禁
            allowed_tools=["Read", "Grep", "Glob"],
        ),
    ):
        if isinstance(message, ResultMessage) and message.subtype == "success":
            print(message.result)


await demo_skills()

### 20.3 Slash commands

把 `/compact`、`/clear` 或自定义命令**直接当 prompt 字符串发**即可（这类命令是 SDK 输入，不是 CLI 专属）：

```python
# 可用命令清单在 init 消息里。§5「最小可运行示例」 那次真实运行的 init 消息里是：
# 'slash_commands': ['compact', 'context', 'cost', 'init', 'pr-comments',
#                    'release-notes', 'review', 'security-review']
async for m in query(prompt="/compact", options=ClaudeAgentOptions(continue_conversation=True)):
    ...  # 压缩结果读 compact_boundary system 消息的 data["compact_metadata"]
```

作用于对话历史的命令（`/compact`）要发到已有历史的会话（`continue_conversation=True` 或 streaming 模式）。自定义命令的 `.claude/commands/*.md` 是**遗留格式**，新命令推荐直接写成 skill（同样支持 `/name` 调用，还多了自主触发）。


### 20.4 Plugins

Plugin = 一包可跨项目分发的扩展（skills + agents + hooks + MCP servers）：

```python
options = ClaudeAgentOptions(
    plugins=[{"type": "local", "path": "./my-plugin"}]  # type 只接受 "local"
)
```

path 指向 plugin 根目录（`skills/`、`agents/`、`hooks/`、`.claude-plugin/` 的父目录）。plugin 的 skill/命令自动加 `plugin-name:` 前缀防冲突，调用发 `/plugin-name:skill-name`；加载验证读 init 消息的 `data["plugins"]` / `data["skills"]`。


### 20.5 特性选型速查

| 目标 | 用什么 |
|---|---|
| 项目约定 agent 一直遵守 | CLAUDE.md（`setting_sources` 含 `"project"`） |
| 按需加载的参考材料 / 可复用工作流 | Skills |
| 隔离子任务、新 context | Subagents（§14「子 agent」） |
| tool call 上的确定性逻辑 | Hooks（§11） |
| 接外部服务 | MCP（§12「外部 MCP」） |
| 打包分发以上全部 | Plugins |